# Cost-Sensitive Machine Learning for Telecom Value Compression
**Repository:** [`see0mar/cost-sensitive-telecom-retention`](https://github.com/see0mar/cost-sensitive-telecom-retention)  
**Author:** Omar ElTokhy · GCI World — Matsuo Iwasawa Lab, Tokyo University  
**Stack:** Python 3.10 · LightGBM · scikit-learn · SHAP · 1 M-record synthetic benchmark

---

## Problem Statement

In subscription-based telecom markets, a subscriber experiencing service friction
may quietly **reduce their monthly spend by ≥ 25 %** — downgrading packages,
dropping data add-ons, or cutting wallet reloads — while their line remains
**fully active**. Standard CRM triggers miss this cohort entirely.

This notebook demonstrates a complete ML pipeline that:

1. Detects value-compressed subscribers using six behavioural features.
2. Benchmarks three algorithms under identical conditions.
3. Replaces the default 0.50 probability threshold with a **cost-sensitive
   boundary derived from the asymmetric cost matrix** (offer cost C = 50 EGP,
   save value V = 500 EGP → θ* = C/V = **0.10**).
4. Demonstrates the financial lift: **+373,950 EGP** net campaign value
   from a single parameter change.

---

### Analytical Hypotheses

| # | Hypothesis | Feature |
|---|------------|---------|
| H1 | Recurring support-call frequency is a **leading indicator** of spend restriction | `customer_service_calls_mean` |
| H2 | Short-term **revenue velocity** (ΔR ≤ −0.25) is a stronger signal than long-term billing averages | `monthly_overage_fee_mean` |
| H3 | A cost-sensitive decision boundary outperforms any symmetric accuracy metric | threshold optimisation |


## 0 · Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import lightgbm as lgb
import shap
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    confusion_matrix, classification_report,
)

from src.data_processor  import TelemetryDataProcessor
from src.model_trainer   import RetentionModelTrainer
from src.cost_evaluator  import CostSensitiveOptimizer

# ── Visual config ──────────────────────────────────────────────────────────
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
sns.set_theme(style='whitegrid', palette='muted')

C_STABLE     = '#1E3A8A'   # dark blue — stable accounts
C_COMPRESSED = '#DC2626'   # red       — compressed accounts
C_CHAMPION   = '#15803D'   # green     — LightGBM / optimised
C_NEUTRAL    = '#6B7280'   # grey      — baselines
C_GOLD       = '#B45309'   # amber     — highlights

DATA_PATH = '../data/synthetic_telecom_retention_1m.parquet'

print('Environment ready.')


## 1 · Synthetic Benchmark Dataset

> The dataset is generated by `scripts/generate_data.py`.
> If the Parquet file does not exist yet, the cell below will create it.


In [ ]:
if not os.path.exists(DATA_PATH):
    import subprocess
    print('Generating dataset — this takes ~1–2 seconds...')
    subprocess.run(['python', '../scripts/generate_data.py'], check=True)

processor = TelemetryDataProcessor(DATA_PATH)
df = processor.load_data()
df.head(3)


In [ ]:
# ── Class balance & memory footprint ──────────────────────────────────────
n_total   = len(df)
n_pos     = int(df['value_compressed_flag'].sum())
n_neg     = n_total - n_pos
pos_rate  = n_pos / n_total
mem_mb    = df.memory_usage(deep=True).sum() / 1024**2

print(f'Records       : {n_total:>10,}')
print(f'Stable (0)    : {n_neg:>10,}  ({1-pos_rate:.1%})')
print(f'Compressed (1): {n_pos:>10,}  ({pos_rate:.1%})')
print(f'Class ratio   : {n_neg/n_pos:.1f} : 1  → scale_pos_weight ≈ {n_neg/n_pos:.2f}')
print(f'Memory in RAM : {mem_mb:.1f} MB')


## 2 · Exploratory Data Analysis

Three EDA outputs guide feature selection and validate the hypotheses
before any model is trained.


In [ ]:
# ── EDA 1: Class balance bar ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left — class distribution
counts = df['value_compressed_flag'].value_counts().sort_index()
axes[0].bar(['Stable (0)', 'Compressed (1)'], counts.values,
            color=[C_STABLE, C_COMPRESSED], width=0.45, edgecolor='white')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 8000, f'{v:,}\n({v/n_total:.1%})',
                ha='center', fontsize=9.5, fontweight='bold')
axes[0].set_title('Class Distribution (4 : 1 Imbalance)', fontweight='bold')
axes[0].set_ylabel('Subscriber Count')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

# Right — tenure by class
for label, color, name in [(0, C_STABLE, 'Stable'), (1, C_COMPRESSED, 'Compressed')]:
    axes[1].hist(df.loc[df['value_compressed_flag']==label, 'account_tenure_months'],
                 bins=35, alpha=0.55, color=color, label=name, density=True)
axes[1].set_title('Tenure Distribution by Class  (H3 Validation)', fontweight='bold')
axes[1].set_xlabel('Account Tenure (months)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('Dataset Overview', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print('H3 finding: tenure distributions overlap — long-tenure customers are NOT immune.')


In [ ]:
# ── EDA 2: Hypothesis 1 — Support-call inflection boundary ────────────────
fig, ax = plt.subplots(figsize=(9, 4))

for label, color, name in [(0, C_STABLE, 'Stable'), (1, C_COMPRESSED, 'Compressed')]:
    sns.kdeplot(
        data=df[df['value_compressed_flag'] == label],
        x='customer_service_calls_mean',
        ax=ax, color=color, fill=True, alpha=0.25, label=name, common_norm=False
    )

INFLECTION = 3.89
ax.axvline(INFLECTION, color=C_COMPRESSED, linestyle='--', linewidth=1.8,
           label=f'Inflection Threshold ({INFLECTION} calls/month)')
ax.text(INFLECTION + 0.15, ax.get_ylim()[1] * 0.85,
        f'θ_service = {INFLECTION}\nRisk sharply\nincreases →',
        fontsize=8.5, color=C_COMPRESSED)

ax.set_title('H1: Support Contact Density — Stable vs. Compressed',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Monthly Customer-Service Calls')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()
print('H1 validated: Compressed subscribers cluster strongly above the 3.89-call threshold.')


In [ ]:
# ── EDA 3: Feature Correlation Matrix ─────────────────────────────────────
feature_cols = [
    'monthly_overage_fee_mean', 'data_overage_fee_mean',
    'dropped_blocked_calls_mean', 'customer_service_calls_mean',
    'estimated_income_bracket', 'account_tenure_months',
]
corr = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdBu_r',
    vmin=-1, vmax=1, center=0,
    square=True, linewidths=0.4,
    xticklabels=[c.replace('_mean','').replace('_',' ') for c in feature_cols],
    yticklabels=[c.replace('_mean','').replace('_',' ') for c in feature_cols],
    ax=ax
)
ax.set_title('Feature Correlation Matrix  (No Severe Multicollinearity)',
             fontweight='bold')
plt.tight_layout()
plt.show()
print('All 6 features retained — highest correlation: dropped_blocked ↔ customer_service (expected).')


## 3 · Stratified Train / Test Split

A **70 / 30 stratified split** is applied via `TelemetryDataProcessor`.
Identifier columns (`customer_id`) are automatically excluded to prevent
data leakage. Retrospective aggregate columns (`change_rev`, `avg3rev`, …)
are filtered out at load time.


In [ ]:
X_train, X_test, y_train, y_test = processor.prepare_splits(
    test_size=0.30, random_state=42
)

print(f'Features used : {list(X_train.columns)}')
print(f'Train positives: {y_train.mean():.4f}  |  Test positives: {y_test.mean():.4f}')
print(f'Class ratio in test set: {(y_test==0).sum()} : {(y_test==1).sum()}')


## 4 · Three-Model Benchmark

| Model | Role | Selection Rationale |
|-------|------|---------------------|
| Logistic Regression | Linear baseline | Full interpretability; performance ceiling test |
| Decision Tree (depth=6) | Rule extraction | Identifies natural split boundaries (e.g. 3.89-call threshold) |
| **LightGBM** | **Champion** | Leaf-wise growth; native `scale_pos_weight`; Optuna-compatible |


In [ ]:
trainer = RetentionModelTrainer(random_state=42)

# Train baselines
baselines = trainer.train_baselines(X_train, y_train)

# Train champion (scale_pos_weight computed dynamically from y_train)
lgb_model = trainer.train_lightgbm(X_train, y_train)


In [ ]:
# ── AUC Benchmark Table ────────────────────────────────────────────────────
model_registry = {
    'Logistic Regression': baselines['logistic_regression'],
    'Decision Tree':       baselines['decision_tree'],
    'LightGBM (Champion)': lgb_model,
}

probs_registry = {}
print('\n  Model Benchmark Results')
print('  ' + '─' * 52)
for name, model in model_registry.items():
    probs = trainer.predict_proba(model, X_test)
    auc   = roc_auc_score(y_test, probs)
    probs_registry[name] = probs
    marker = '  ← CHAMPION' if 'LightGBM' in name else ''
    print(f'  {name:<30}  ROC-AUC: {auc:.4f}{marker}')
print('  ' + '─' * 52)


## 5 · Model Diagnostics

### 5a · Confusion Matrices (test set, n = 300,000)

Each matrix uses the **default 0.50 threshold** — which is intentionally
sub-optimal given the asymmetric cost structure. Section 6 corrects this.


In [ ]:
THRESHOLD_DEFAULT = 0.50
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

cm_colors = {
    'Logistic Regression': 'Blues',
    'Decision Tree':       'Purples',
    'LightGBM (Champion)': 'Greens',
}

for ax, (name, probs) in zip(axes, probs_registry.items()):
    preds = (probs >= THRESHOLD_DEFAULT).astype(int)
    cm    = confusion_matrix(y_test, preds)
    sns.heatmap(
        cm, annot=True, fmt=',', cmap=cm_colors[name],
        xticklabels=['Stable (0)', 'Compressed (1)'],
        yticklabels=['Stable (0)', 'Compressed (1)'],
        ax=ax, linewidths=0.5, linecolor='white'
    )
    auc = roc_auc_score(y_test, probs)
    ax.set_title(f'{name}\nAUC = {auc:.4f}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.suptitle(f'Confusion Matrices at Default Threshold (θ = {THRESHOLD_DEFAULT})',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### 5b · ROC Curve Comparison

The AUC measures discriminatory power **across all thresholds** — confirming
LightGBM's advantage is not an artefact of threshold choice.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

roc_styles = {
    'Logistic Regression': (C_STABLE,     '--', 1.5),
    'Decision Tree':       ('#7C3AED',    ':',  1.5),
    'LightGBM (Champion)': (C_CHAMPION,   '-',  2.2),
}

for name, probs in probs_registry.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc          = roc_auc_score(y_test, probs)
    color, ls, lw = roc_styles[name]
    ax.plot(fpr, tpr, color=color, linestyle=ls, linewidth=lw,
            label=f'{name}  (AUC = {auc:.4f})')

ax.plot([0,1],[0,1], 'k:', linewidth=0.8, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve — Three-Model Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 5c · Champion Model — Full Classification Report

In [ ]:
lgb_preds_default = (probs_registry['LightGBM (Champion)'] >= THRESHOLD_DEFAULT).astype(int)
print('LightGBM at θ = 0.50 (default):')
print(classification_report(
    y_test, lgb_preds_default,
    target_names=['Stable (0)', 'Compressed (1)']
))


## 6 · Cost-Sensitive Threshold Optimisation

### The NCV Maximisation Formula

$$\text{NCV}(\theta) = TP(\theta) \cdot (V - C) - FP(\theta) \cdot C$$

| Symbol | Value | Meaning |
|--------|-------|---------|
| V | 500 EGP | Gross value preserved per correctly retained subscriber |
| C |  50 EGP | Cost of one proactive retention offer |
| θ* = C/V | **0.10** | Break-even threshold (any subscriber above 10% risk probability is worth targeting) |

> Scanning all thresholds 0.01 → 0.99 and selecting the one that **maximises
> net business value** rather than raw accuracy.


In [ ]:
optimizer = CostSensitiveOptimizer(
    value_per_save_egp=500.0,
    cost_per_offer_egp=50.0,
)

lgb_probs = probs_registry['LightGBM (Champion)']

best_threshold, results_df = optimizer.optimize_threshold(
    y_true=y_test.values,
    y_probs=lgb_probs,
)


In [ ]:
# ── NCV curve plot ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
optimizer.plot_ncv_curve(results_df, best_threshold, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
# ── Financial comparison table ─────────────────────────────────────────────
summary = optimizer.financial_summary_table(results_df, best_threshold)

print('\n  Financial Simulation Ledger  (LightGBM Champion)')
print('  ' + '═' * 74)
print(summary.to_string())
print('  ' + '═' * 74)

# Compute lift
base_ncv = float(results_df.loc[
    results_df['threshold'].sub(0.50).abs().idxmin(),
    'net_campaign_value_egp'
])
opt_ncv = float(results_df.loc[
    results_df['threshold'].sub(best_threshold).abs().idxmin(),
    'net_campaign_value_egp'
])
lift = opt_ncv - base_ncv
mult = opt_ncv / base_ncv if base_ncv > 0 else float('inf')

print(f'\n  Incremental net value lift : +{lift:,.0f} EGP')
print(f'  Net value multiplier       :  ×{mult:.1f}')
print(f'  Same data · same model · single parameter shift.')


## 7 · Model Explainability — SHAP

SHAP (SHapley Additive exPlanations) provides **model-agnostic feature
importance** that confirms the directional hypotheses are correctly
encoded in the LightGBM champion model.

- **Red** = high feature value · **Blue** = low feature value
- Rightward displacement = pushes risk **up** (towards compression)
- Leftward displacement = pushes risk **down** (towards stability)


In [ ]:
# ── SHAP on a 2,000-record sample (for speed) ──────────────────────────────
SHAP_SAMPLE = 2000
X_shap = X_test.iloc[:SHAP_SAMPLE]

explainer   = shap.TreeExplainer(lgb_model)
explanation = explainer(X_shap)  # returns Explanation object (modern SHAP API)

print(f'SHAP values computed for {SHAP_SAMPLE:,} test records.')
print(f'Explanation values shape: {explanation.values.shape}')


In [ ]:
# ── Global bar plot ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plt.sca(axes[0])
shap.plots.bar(explanation, max_display=6, show=False)
axes[0].set_title('Mean |SHAP| — Global Feature Importance',
                  fontweight='bold', fontsize=11)

# ── Beeswarm / summary plot ────────────────────────────────────────────────
plt.sca(axes[1])
shap.plots.beeswarm(explanation, max_display=6, show=False)
axes[1].set_title('Directional SHAP — Feature Effect on Risk Score',
                  fontweight='bold', fontsize=11)

plt.suptitle('LightGBM Champion — SHAP Explainability (n = 2,000)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Local waterfall — single flagged account ───────────────────────────────
# Find the first account flagged as high-risk at the optimised threshold
flagged_idx = np.where(lgb_probs[:SHAP_SAMPLE] >= best_threshold)[0]
example_idx = int(flagged_idx[0]) if len(flagged_idx) > 0 else 0

fig, ax = plt.subplots(figsize=(9, 4))
plt.sca(ax)
shap.plots.waterfall(explanation[example_idx], max_display=6, show=False)
ax.set_title(
    f'Local SHAP Waterfall — Subscriber #{example_idx} '
    f'(P = {lgb_probs[example_idx]:.3f}  |  threshold = {best_threshold:.2f})',
    fontweight='bold', fontsize=10
)
plt.tight_layout()
plt.show()

print('Interpretation: each bar shows how much that feature pushed the risk score')
print('above or below the dataset\'s mean prediction.')


## 8 · Results Summary

| Metric | Value |
|--------|-------|
| Champion model | LightGBM (leaf-wise GBDT) |
| ROC-AUC (held-out test) | 0.8379 |
| Class imbalance ratio | ≈ 4 : 1 (handled via `scale_pos_weight`) |
| Default θ = 0.50  — Net Campaign Value | **17,700 EGP** |
| Optimised θ = 0.10 — Net Campaign Value | **391,650 EGP** |
| Net value lift | **+373,950 EGP  (×22)** |

---

**Key finding:** the dominant performance driver is not model architecture
but **decision-boundary engineering** — aligning the classification threshold
to the true cost ratio (C/V = 50/500 = 0.10) rather than the symmetric
accuracy optimum (0.50).

*All figures are simulated estimates based on the stated cost assumptions.
Real-world results depend on model calibration, intervention effectiveness,
and actual unit economics.*
